In [1]:
import plotly.express as px

from pull_data import load_draft_dataframe

In [2]:
# creates the dataframe by calling the function from 'pull_data.py'
# the function in 'pull_data.py':
    # 1. Downloads the draft selections
    # 2. Downloads leage rosters and results
    # 3. Matches each selection to its roster
    # 4. Creates pick_in_round which determines which round a player was drafted in
    # 5. Classifies each selection in a round as 'Early', 'Middle', or 'Late'
        # 5a. Early = picks 1-4
        # 5b. Middle = picks 5-8
        # 5c. Late = picks 9-12
    # 6. Returns the completed DataFrame

df = load_draft_dataframe()

df.head()

,pick_no,pick_in_round,round,player_id,player,position,roster_id,team_points,potential_points,team_wins,team_losses,pick_group
0,1,1,1,6820,Clyde Edwards-Helaire,RB,4,1440,1823,8,5,Early
1,2,2,1,6813,Jonathan Taylor,RB,5,1280,1584,6,7,Early
2,3,3,1,6806,J.K. Dobbins,RB,11,1414,1763,8,5,Early
3,4,4,1,6783,Jerry Jeudy,WR,2,887,1019,1,12,Early
4,5,5,1,6938,Cam Akers,RB,8,1185,1557,7,6,Middle


In [3]:
# Checks to see if data has any missing data that needs to be addressed
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

# returns the 6 columns that are the most relevant and shows the first 12 rows
df[
    [
        "round",
        "pick_in_round",
        "pick_group",
        "player",
        "position",
        "team_points",
        "team_wins",
    ]
].head(12)

Dataset shape: (48, 12)

Missing values:
pick_no             0
pick_in_round       0
round               0
player_id           0
player              0
position            0
roster_id           0
team_points         0
potential_points    0
team_wins           0
team_losses         0
pick_group          0
dtype: int64


,round,pick_in_round,pick_group,player,position,team_points,team_wins
0,1,1,Early,Clyde Edwards-Helaire,RB,1440,8
1,1,2,Early,Jonathan Taylor,RB,1280,6
2,1,3,Early,J.K. Dobbins,RB,1414,8
3,1,4,Early,Jerry Jeudy,WR,887,1
4,1,5,Middle,Cam Akers,RB,1185,7
5,1,6,Middle,D'Andre Swift,RB,887,1
6,1,7,Middle,CeeDee Lamb,WR,887,1
7,1,8,Middle,Justin Jefferson,WR,887,1
8,1,9,Late,Jalen Reagor,WR,1258,8
9,1,10,Late,Henry Ruggs III,WR,1243,4


# Draft Position Analysis

## Questions

1. Are earlier draft selections associated with more team points?
2. Are earlier draft selections associated with more team wins?
3. Are later draft selections associated with fewer team points and wins?

In [4]:
# Divides the groups by the round and by the pick group (Early, Middle, Late)
# Aggregations:
    # average_team_points: the average points earned by teams who had picks 
    # in different pick groups in each round

    # median_team_points: the median amount of points earned by teams who had picks 
    # in different pick groups in each round

    # number_of_picks: the number of picks that were made in each pick group by round

# DataFrame is created using points_summary and displays all the info. above

points_summary = (
    df.groupby(
        ["round", "pick_group"],
        observed=True,
    )
    .agg(
        average_team_points=("team_points", "mean"),
        median_team_points=("team_points", "median"),
        number_of_picks=("pick_no", "count"),
    )
    .reset_index()
)

points_summary

,round,pick_group,average_team_points,median_team_points,number_of_picks
0,1,Early,1255.25,1347.0,4
1,1,Middle,961.50,887.0,4
2,1,Late,1068.75,1065.0,4
3,2,Early,1342.50,1336.0,4
4,2,Middle,1332.00,1337.5,4
5,2,Late,1341.75,1337.5,4
6,3,Early,1302.50,1269.0,4
7,3,Middle,1270.75,1229.0,4
8,3,Late,1422.25,1427.0,4
9,4,Early,1269.50,1270.0,4


In [5]:
# Creates a combined bar graph that displays the average points teams in the 3 pick 
# groups had in all 4 rounds

fig = px.bar(
    points_summary,
    x="round",
    y="average_team_points",
    color="pick_group",
    barmode="group",
    category_orders={
        "pick_group": ["Early", "Middle", "Late"]
    },
    labels={
        "round": "Draft Round",
        "average_team_points": "Average Team Points",
        "pick_group": "Selection Group",
    },
    title="Average Team Points by Round and Selection Group",
)

fig.show()

In [6]:
# Creates a combined scatter plot where each point represents the player drafted along
# with their team points, position, and pick selected

# The regression line shows the overall direction of the relationship and the correlation
# of the regression line and the data points in the individual pick group

fig = px.scatter(
    df,
    x="pick_in_round",
    y="team_points",
    color="pick_group",
    facet_col="round",
    facet_col_wrap=2,
    trendline="ols",
    hover_data=[
        "player",
        "position",
        "roster_id",
        "pick_no",
    ],
    category_orders={
        "pick_group": ["Early", "Middle", "Late"]
    },
    labels={
        "pick_in_round": "Pick Within Round",
        "team_points": "Team Points",
        "pick_group": "Selection Group",
    },
    title="Pick Position and Team Points Within Each Round",
)

fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.replace("round=", "Round ")
    )
)

fig.show()

In [7]:
# Calculates the correlation between individual draft picks in each pick group for each
# round
# In the graph, represented by the regression line

# Negative correlation: earlier picks are associated with more points.
# Positive correlation: later picks are associated with more points.
# Correlation near zero: little apparent relationship.

points_correlations = (
    df.groupby("round")
    .apply(
        lambda group: group["pick_in_round"].corr(
            group["team_points"]
        ),
        include_groups=False,
    )
    .rename("points_correlation")
    .reset_index()
)

points_correlations

,round,points_correlation
0,1,-0.545282
1,2,0.045996
2,3,0.435798
3,4,0.037411


## Does having an earlier selection in the fantasy draft allow for more points to be obtained by a team?

- The relationship between selection position and team points varied between rounds. In round 1, early selections had the highest team points with 1255.25, and a downward sloping regression line with a moderately negative correlation of -0.5453 suggests that as the picks progressed later through the rounds, the association with team points decreased. Round 2 had nearly identical results between the different pick groups with an extremely weak correlation of 0.046. This suggests that Early picks in round 2 don't have as much of an influence on team points than Middle or Late selections. In round 3, Late selections had the highest team points with 1422.25. With a moderately positive regression line and a correlation of 0.4358, the data suggests that the average team points is moderately associated with selections in the Late portion of the round. Finally, in round 4, Middle selections had the highest team points with 1342.5. However, a nearly flat regression line along with a correlation value of approximately 0 (0.0374) represents that there isn't a relationship between where a player was drafted in the round and the amount of team points that came from it.

## Team Wins

The following analysis compares early, middle, and late picks with the
number of wins earned by the team that made each selection.

In [8]:
# Divides the groups by the round and by the pick group (Early, Middle, Late) again
# Aggregations:
    # average_wins: the average wins by teams who had picks 
    # in different pick groups in each round

    # median_wins: the median number of wins by teams who had picks 
    # in different pick groups in each round

    # number_of_picks: the number of picks that were made in each pick group by round

# DataFrame is created using wins_summary and displays all the info. above

wins_summary = (
    df.groupby(
        ["round", "pick_group"],
        observed=True,
    )
    .agg(
        average_wins=("team_wins", "mean"),
        median_wins=("team_wins", "median"),
        number_of_picks=("pick_no", "count"),
    )
    .reset_index()
)

wins_summary

,round,pick_group,average_wins,median_wins,number_of_picks
0,1,Early,5.75,7.0,4
1,1,Middle,2.50,1.0,4
2,1,Late,3.50,2.5,4
3,2,Early,8.00,8.0,4
4,2,Middle,8.00,8.5,4
5,2,Late,7.25,7.0,4
6,3,Early,7.50,8.0,4
7,3,Middle,8.00,7.5,4
8,3,Late,8.25,8.0,4
9,4,Early,7.50,8.0,4


In [9]:
# Creates a combined bar chart that displays the average number of team wins based on
# which pick group and round each team selected

# Because this draft was a snake draft (first person in round 1 picks last in round 2
# then picks first again in round 3, etc.) the teams selecting early changes with each 
# round

fig = px.bar(
    wins_summary,
    x="round",
    y="average_wins",
    color="pick_group",
    barmode="group",
    category_orders={
        "pick_group": ["Early", "Middle", "Late"]
    },
    labels={
        "round": "Draft Round",
        "average_wins": "Average Team Wins",
        "pick_group": "Selection Group",
    },
    title="Average Team Wins by Round and Selection Group",
)

fig.show()

In [10]:
# Creates a combined scatterplot that displays the pick a player was selected, the pick
# group that player is in, and how many team wins the team that selected that player had

# Generated a regression line that shows the correlation between a player's pick number
# and how many wins a team got

fig = px.scatter(
    df,
    x="pick_in_round",
    y="team_wins",
    color="pick_group",
    facet_col="round",
    facet_col_wrap=2,
    trendline="ols",
    hover_data=[
        "player",
        "position",
        "roster_id",
        "team_points",
    ],
    category_orders={
        "pick_group": ["Early", "Middle", "Late"]
    },
    labels={
        "pick_in_round": "Pick Within Round",
        "team_wins": "Team Wins",
        "pick_group": "Selection Group",
    },
    title="Pick Position and Team Wins Within Each Round",
)

fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.replace("round=", "Round ")
    )
)

fig.show()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2018: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr / self.centered_tss


In [11]:
# Calculates the correlation between the pick number a team had and how many wins the
# team got during the season

# Once again: 
    # Negative: earlier selections were associated with more wins.
    # Positive: later selections were associated with more wins.
    # Near zero: draft position and wins had little linear relationship.

wins_correlations = (
    df.groupby("round")
    .apply(
        lambda group: group["pick_in_round"].corr(
            group["team_wins"]
        ),
        include_groups=False,
    )
    .rename("wins_correlation")
    .reset_index()
)

wins_correlations

,round,wins_correlation
0,1,-0.518755
1,2,-0.055381
2,3,0.215132
3,4,-0.226923


## Does having an earlier selection in the fantasy draft allow for more points to be obtained by a team?

- The relationship between selection position and team wins also varied between rounds. In round 1, Early selections had the highest team wins with 5.75, and a downward sloping regression line with a moderately negative correlation of -0.5188 suggests that as the pick position became later within round 1, team wins generally decreased. Round 2 had identical results between the Early and Middle picks with 8 wins while Late picks had lower results with 7.25. This suggests that selection position had almost no linear association with team points in round 2, however, as the correlation is -0.0554. In round 3, Late selections also had the highest team wins with 8.25. With a weakly positive regression line and a correlation of 0.2151, the data suggests that average team wins isn't consistently associated with different round groups. Finally, in round 4, Early and Middle selections were once again identical while Late selections were lower. Early and Middle selections had an average of 7.5 wins while Late had 6. Additionally, the downward-sloping regression line along with the weak negative correlation of -0.2269 imply that while Early selections in the round have higher team wins, it isn't very strong.

In [12]:
# Creates one combined table showing the overall information collected from above
# Displays average team points, average wins, pick group, and round selected

analysis_summary = points_summary.merge(
    wins_summary,
    on=["round", "pick_group"],
    suffixes=("_points", "_wins"),
)

analysis_summary[
    [
        "round",
        "pick_group",
        "average_team_points",
        "average_wins",
        "number_of_picks_points",
    ]
].round(2)

,round,pick_group,average_team_points,average_wins,number_of_picks_points
0,1,Early,1255.25,5.75,4
1,1,Middle,961.50,2.50,4
2,1,Late,1068.75,3.50,4
3,2,Early,1342.50,8.00,4
4,2,Middle,1332.00,8.00,4
5,2,Late,1341.75,7.25,4
6,3,Early,1302.50,7.50,4
7,3,Middle,1270.75,8.00,4
8,3,Late,1422.25,8.25,4
9,4,Early,1269.50,7.50,4


## Are earlier draft selections associated with more team wins?

- Later selections were not consistently associated with fewer team points or wins. In round 1, the Early group had higher average points and wins than the Middle and Late groups. This pattern mostly disappeared in round 2. In round 3, the pattern reversed, as the Late group had the highest average points and wins. Round 4 produced mixed results: the Middle group had the highest average points, while the Early and Middle groups tied for the highest average wins. Overall, later selections were not consistently associated with worse team performance.

# Conclusions

- Overall, the findings from this league do not show consistent evidence that earlier draft selections correspond to more team points and team wins. Earlier selections were associated with more team points and team wins in round 1, but the pattern mostly vanished throughout the other rounds. Therefore, these results suggest that the relationship between selection position and team performance mostly depends on the round being examined.

- These findings show associations, not causation. This analysis covers only one league with each selection group containing only four picks per round. Additionally, team points and wins represent the performance of the entire roster rather than the individual player selected. In order to draw stronger conclusions about this trend, a larger dataset from multiple leagues and drafts is required. This is something that I would be interested in expanding on further while using more advanced techniques to gather and analyze data.
